In [2]:
import numpy as np
import pandas as pd
import xarray as xr

In [3]:
# static features
stat_dict = {}
stat_features = ['clelev', 'clfaotxt', 'cltveg', 'phh2o', 'topsoil_bulk_density', 'vegetation_type_11', 'wetlandtype', 'climatetype']
stat_files = ['clelev.nc', 'clfaotxt.nc', 'cltveg.nc', 'phh2o.nc', 'topsoil_bulk_density.nc', 'vegetation_type_11.nc', 'wetlandtype.nc', 'climatetype.nc'] 
for i, feat in enumerate(stat_features):
    filename = f'../../data/TEM-MDM/{stat_files[i]}' 
    print(filename)
    feature = xr.open_dataset(filename)
    feature = feature[feat].values
    stat_dict[feat] = feature

../../data/TEM-MDM/clelev.nc
../../data/TEM-MDM/clfaotxt.nc
../../data/TEM-MDM/cltveg.nc
../../data/TEM-MDM/phh2o.nc
../../data/TEM-MDM/topsoil_bulk_density.nc
../../data/TEM-MDM/vegetation_type_11.nc
../../data/TEM-MDM/wetlandtype.nc
../../data/TEM-MDM/climatetype.nc


In [4]:
# non-spatial features
nosp_features = ['ch4','co2']
nosp_files = ['ch4-1979-2018','kco21979-2018']
ch4_dict, co2_dict = {}, {}
with open('../../data/TEM-MDM/ch4-1979-2018.txt', 'r') as f:
    for line in f:
        year, ch4 = line.split()
        ch4_dict[int(year)] = float(ch4)
with open('../../data/TEM-MDM/kco21979-2018.txt', 'r') as f:
    for line in f:
        year, co2 = line.split()
        co2_dict[int(year)] = float(co2)

In [5]:
def convert2year(data):
    days_in_month = [31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
    mask = np.zeros(data.shape, dtype=bool)
    for month, days in enumerate(days_in_month):
        mask[:, :, month, :days] = True 
    valid_data = data[mask]
    valid_data = valid_data.reshape(data.shape[0], data.shape[1], 365)
    return valid_data

In [6]:
site_info = pd.read_csv('../../data/FLUXNET-CH4/FLUXNET_CH4_2024.csv')
fluxnet_data = pd.read_csv('../../data/FLUXNET-CH4/FLUXNET_T1_DD.csv')

In [7]:
location = {}
for index, line in site_info.iterrows():
    location[line['SITE_ID'].replace('-','.')] = (line['LOCATION_LAT'], line['LOCATION_LONG'], line["LOCATION_ELEV"], line["IGBP"], line["MAT"], line['MAP'])

print(len(location))
print(location.keys())
print('ch4_t1' in location.keys())

79
dict_keys(['AT.Neu', 'BR.Npw', 'BW.Gum', 'BW.Nxr', 'CA.SCB', 'CA.SCC', 'CH.Cha', 'CH.Dav', 'CH.Oe2', 'CN.Hgu', 'DE.Dgw', 'DE.Hte', 'DE.SfN', 'DE.Zrk', 'FI.Hyy', 'FI.Lom', 'FI.Si2', 'FI.Sii', 'FR.LGt', 'HK.MPM', 'ID.Pag', 'IT.BCi', 'IT.Cas', 'JP.BBY', 'JP.Mse', 'JP.SwL', 'KR.CRK', 'MY.MLM', 'NL.Hor', 'NZ.Kop', 'PH.RiF', 'RU.Ch2', 'RU.Che', 'RU.Cok', 'RU.Fy2', 'SE.Deg', 'UK.LBT', 'US.A03', 'US.A10', 'US.Atq', 'US.BZB', 'US.BZF', 'US.BZS', 'US.Beo', 'US.Bes', 'US.Bi1', 'US.Bi2', 'US.CRT', 'US.DPW', 'US.EDN', 'US.EML', 'US.HRA', 'US.HRC', 'US.Ho1', 'US.ICs', 'US.Ivo', 'US.LA1', 'US.LA2', 'US.Los', 'US.MAC', 'US.MRM', 'US.Myb', 'US.NC4', 'US.NGB', 'US.NGC', 'US.ORv', 'US.OWC', 'US.PFa', 'US.Snd', 'US.Sne', 'US.Srr', 'US.StJ', 'US.Tw1', 'US.Tw3', 'US.Tw4', 'US.Tw5', 'US.Twt', 'US.Uaf', 'US.WPT'])
False


In [8]:
fluxnet_data['TIMESTAMP'] = pd.to_datetime(fluxnet_data['TIMESTAMP'])
fluxnet_data['Year'] = fluxnet_data['TIMESTAMP'].dt.year
fluxnet_data['TIMESTAMP'] = pd.to_datetime(fluxnet_data['TIMESTAMP'])

fluxnet_data = fluxnet_data[~((fluxnet_data['TIMESTAMP'].dt.month == 2) & (fluxnet_data['TIMESTAMP'].dt.day == 29))]
grouped = fluxnet_data.groupby(['Site', 'Year'])
print('ch4_t1' in fluxnet_data['Site'].values)
for site in location.keys():
    if site not in fluxnet_data['Site'].values:
        print("site without data:", site)

for site, year in grouped.groups.keys():
    if site not in location:
        print("site not in location:", site)

True
site without data: US.MAC
site without data: US.StJ
site without data: US.Tw1
site without data: US.Tw3
site without data: US.Tw4
site without data: US.Tw5
site not in location: ch4_t1
site not in location: ch4_t1
site not in location: ch4_t1
site not in location: ch4_t1
site not in location: ch4_t1
site not in location: ch4_t1
site not in location: ch4_t1
site not in location: ch4_t1
site not in location: ch4_t1
site not in location: ch4_t1
site not in location: ch4_t1
site not in location: ch4_t1
site not in location: ch4_t1


In [15]:
splitting_year_dict = {}
print("len of fluxnet_data:", len(fluxnet_data.groupby('Site')))
for site, group in fluxnet_data.groupby('Site'):
    if site not in location:
        continue
    unique_years = sorted(group['Year'].unique())
    # if unique_years is None or len(unique_years) == 3:
    #     print(site, unique_years, (len(unique_years)-1) * 6 // 7)
    for year in unique_years:
        if group[group['Year'] == year].shape[0] != 365:
            print("insufficient data for site-year:", site, year, group[group['Year'] == year].shape[0])
            unique_years.remove(year)
    if len(unique_years) < 3:
        print("not enough years for training:", site, unique_years)
        continue
    midpoint = (len(unique_years)-1) * 6 // 7
    splitting_year = unique_years[midpoint] if unique_years else None
    splitting_year_dict[site] = splitting_year

print(len(splitting_year_dict))
print(splitting_year_dict)

len of fluxnet_data: 74
insufficient data for site-year: BR.Npw 2013 26
not enough years for training: BW.Gum [np.int32(2018)]
not enough years for training: BW.Nxr [np.int32(2018)]
insufficient data for site-year: CA.SCB 2017 321
insufficient data for site-year: CA.SCC 2016 242
not enough years for training: CH.Dav [np.int32(2016), np.int32(2017)]
not enough years for training: CH.Oe2 [np.int32(2018)]
insufficient data for site-year: CN.Hgu 2015 229
not enough years for training: CN.Hgu [np.int32(2016), np.int32(2017)]
not enough years for training: FI.Hyy [np.int32(2016)]
not enough years for training: FR.LGt [np.int32(2017), np.int32(2018)]
insufficient data for site-year: ID.Pag 2016 214
not enough years for training: ID.Pag [np.int32(2017)]
not enough years for training: IT.BCi [np.int32(2017), np.int32(2018)]
not enough years for training: IT.Cas [np.int32(2009), np.int32(2010)]
not enough years for training: JP.Mse [np.int32(2012)]
not enough years for training: JP.SwL [np.int32

In [16]:
def nearest_patch_point(lat, lon):
    lat_index = round((lat + 90) * 2) 
    lon_index = round((lon + 180) * 2)  

    return (lat_index, lon_index)

In [17]:
site_wet = {
    "CA.SCB": 2,
    "CA.SCC": 1,
    "DE.Hte": 2,
    "DE.Zrk": 2,
    "DE.SfN": 2,
    "FI.Lom": 2,
    "FI.Si2": 2,
    "FI.Sii": 2,
    "FR.LGt": 2,
    "JP.BBY": 2,
    "NZ.Kop": 2,
    "RU.Ch2": 2,
    "RU.Cok": 2,
    "SE.Deg": 2,
    "US.A03": 2,
    "US.ICs": 2,
    "US.A10": 2,
    "US.Atq": 2,
    "US.Beo": 2,
    "US.Bes": 2,
    "US.NGB": 2,
    "US.BZB": 1,
    "US.BZF": 2,
    "US.Uaf": 1,
    "US.Ivo": 2,
    "US.Los": 2,
    "US.NGC": 2,
    "BR.Npw": 3,
    "BW.Gum": 4,
    "BW.Nxr": 5,
    "ID.Pag": 3,
    "MY.MLM": 3,
    "US.NC4": 3,
    "US.DPW": 4,
    "US.LA2": 4,
    "US.Myb": 4,
    "US.ORv": 3,
    "US.OWC": 4,
    "US.Sne": 4,
    "US.Tw1": 4,
    "US.Tw4": 4,
    "US.Tw5": 4,
    "US.WPT": 4
}

print(len(site_wet))
counter = 0
array = []
for site in site_wet:
    if site in splitting_year_dict.keys():
        counter += 1
        array.append(site)
print(counter)
print(array)

43
30
['CA.SCB', 'CA.SCC', 'DE.Hte', 'DE.Zrk', 'DE.SfN', 'FI.Lom', 'FI.Si2', 'FI.Sii', 'JP.BBY', 'NZ.Kop', 'RU.Ch2', 'RU.Cok', 'SE.Deg', 'US.A03', 'US.A10', 'US.Atq', 'US.Bes', 'US.NGB', 'US.BZB', 'US.BZF', 'US.Uaf', 'US.Ivo', 'US.Los', 'BR.Npw', 'US.NC4', 'US.DPW', 'US.Myb', 'US.ORv', 'US.Sne', 'US.WPT']


In [29]:
site_info = {}
for (site, year), group in grouped:
    if site not in location.keys():
        continue
    if site in site_info.keys():
        continue
    lat, lon, elev, IGBP, mat, map = location[site]
    lat_index, lon_index = nearest_patch_point(lat, lon)
    flatted_index = lat_index * 720 + lon_index

    site_key = f"{site}"
    site_info[site_key] = {
        'lat': lat_index,
        'lon': lat_index,
        'flatted_index': flatted_index,
    }

print(len(site_info))
print(site_info)

73
{'AT.Neu': {'lat': 274, 'lon': 274, 'flatted_index': 197663}, 'BR.Npw': {'lat': 147, 'lon': 147, 'flatted_index': 106087}, 'BW.Gum': {'lat': 142, 'lon': 142, 'flatted_index': 102645}, 'BW.Nxr': {'lat': 141, 'lon': 141, 'flatted_index': 101926}, 'CA.SCB': {'lat': 303, 'lon': 303, 'flatted_index': 218277}, 'CA.SCC': {'lat': 303, 'lon': 303, 'flatted_index': 218277}, 'CH.Cha': {'lat': 274, 'lon': 274, 'flatted_index': 197657}, 'CH.Dav': {'lat': 274, 'lon': 274, 'flatted_index': 197660}, 'CH.Oe2': {'lat': 275, 'lon': 275, 'flatted_index': 198375}, 'CN.Hgu': {'lat': 246, 'lon': 246, 'flatted_index': 177685}, 'DE.Dgw': {'lat': 286, 'lon': 286, 'flatted_index': 206306}, 'DE.Hte': {'lat': 288, 'lon': 288, 'flatted_index': 207744}, 'DE.SfN': {'lat': 276, 'lon': 276, 'flatted_index': 199103}, 'DE.Zrk': {'lat': 288, 'lon': 288, 'flatted_index': 207746}, 'FI.Hyy': {'lat': 304, 'lon': 304, 'flatted_index': 219289}, 'FI.Lom': {'lat': 316, 'lon': 316, 'flatted_index': 227928}, 'FI.Si2': {'lat': 30

In [18]:
import pickle

# 1. 创建一个“以站点为中心”的缓存
# 结构: all_data_cache[site_name][year] = {'input': ..., 'output': ...}
all_data_cache = {}

# 2. 创建一个从站点名到唯一整数ID的映射
# 这将是传递给 AnnualEncoder 的新 "file_idx"
site_to_id_map = {}
current_id = 0

# 3. 创建训练和测试的样本列表
# 结构: [(site_name, year), (site_name, year), ...]
train_samples_list = []
test_samples_list = []

print("Starting data processing for site-centric cache...")

# 循环来自 cell 31
for (site, year), group in grouped:
    if site not in location.keys():
        continue
    
    lat, lon, elev, IGBP, mat, map = location[site]
    if group.shape[0] != 365:
        continue
    if IGBP != 'WET':
        continue
    if site not in splitting_year_dict.keys():
        continue
    temp = group.to_numpy()[:,-6:-3]
    temp = temp.astype(dtype=float)
    lat_ind, lon_ind = nearest_patch_point(lat, lon)
    # print(site, year, lat_ind, lon_ind)
    for i, feat in enumerate(stat_features):
        if i == 0:
            input = np.tile(elev, (365))[..., np.newaxis]
        elif feat == 'wetlandtype':
            if site in site_wet.keys():
                wet = site_wet[site]
            else:
                wet = stat_dict[feat][lat_ind, lon_ind]
            wet = np.tile(wet, (365))[..., np.newaxis]
            input = np.concatenate((input, wet), axis=1)
        else:
            new_input = stat_dict[feat][lat_ind, lon_ind]
            new_input = np.tile(new_input, (365))[..., np.newaxis]
            input = np.concatenate((input, new_input), axis=1)
    ch4_value = np.tile(ch4_dict[year], (365))[..., np.newaxis]
    co2_value = np.tile(co2_dict[year], (365))[..., np.newaxis]
    input = np.concatenate((input, ch4_value), axis=1)
    input = np.concatenate((input, co2_value), axis=1)

    input = np.concatenate((input, temp[:,2][..., np.newaxis]), axis=1) #prec

    # solr 
    input_file = f'../../data/TEM-MDM/daily_ecmwf_SOLR_{year}.nc'
    feature = convert2year(xr.open_dataset(input_file, engine="netcdf4")['SOLR'].values[lat_ind, lon_ind,:,:].reshape(1,1,12,31))
    input = np.concatenate((input, feature.reshape(365,1)), axis=1)

    input = np.concatenate((input, temp[:,0][..., np.newaxis]), axis=1) # tair
    
    T = group['TA_F'].to_numpy()
    VPD = group['VPD_F'].to_numpy()
    vp = (0.6108 * np.exp(17.27*T/(T+237.3)))*10 - VPD
    input = np.concatenate((input, vp[..., np.newaxis]), axis=1) # vapr

    # npp 
    input_file = f'../../data/TEM-MDM/monthly_NPP_{year}.nc'
    feature = xr.open_dataset(input_file, engine="netcdf4")['NPP'].values[lat_ind, lon_ind,:].reshape(1,1,12)
    feature = np.nan_to_num(feature, nan=0.0)
    feature = np.repeat(feature[:, :, :, np.newaxis], 31, axis=3)
    feature = convert2year(feature)
    input = np.concatenate((input, feature.reshape(365,1)), axis=1)
    
    if np.isnan(input).any():
        print('Input nan')
        continue
    label = group['FCH4_F'].to_numpy()
    mask = np.isnan(label)
    label[mask] = group['FCH4_F_ANNOPTLM'].to_numpy()[mask]

    
    # a. 为新站点分配一个唯一的整数ID
    if site not in site_to_id_map:
        site_to_id_map[site] = current_id
        current_id += 1
        
    # b. 将数据存入缓存
    if site not in all_data_cache:
        all_data_cache[site] = {}
        
    all_data_cache[site][year] = {
        'input': input.astype(np.float32), # 确保数据类型正确
        'output': label.astype(np.float32)
    }
    
    # c. 根据你的分割字典，将 (site, year) 元组添加到相应的列表
    if year > splitting_year_dict[site]:
        print("Test: ", site, " year:", year)
        test_samples_list.append((site, year))
    else:
        print("Training: ", site, " year:", year)
        train_samples_list.append((site, year))

print(f"Processing complete.")
print(f"Total unique sites: {len(site_to_id_map)}")
print(f"Total training samples: {len(train_samples_list)}")
print(f"Total test samples: {len(test_samples_list)}")

# 4. 保存所有内容
# 我们使用 pickle 来保存这个复杂的嵌套字典结构
# 注意：这会保存*未标准化*的数据，我们将在 Dataset 中进行标准化

output_dir = '../../processed_data/FLUXNET-CH4/site_centric/'
import os
os.makedirs(output_dir, exist_ok=True)

with open(os.path.join(output_dir, 'all_data_cache.pkl'), 'wb') as f:
    pickle.dump(all_data_cache, f)
    
with open(os.path.join(output_dir, 'site_to_id_map.pkl'), 'wb') as f:
    pickle.dump(site_to_id_map, f)
    
with open(os.path.join(output_dir, 'train_samples.pkl'), 'wb') as f:
    pickle.dump(train_samples_list, f)
    
with open(os.path.join(output_dir, 'test_samples.pkl'), 'wb') as f:
    pickle.dump(test_samples_list, f)


print("Site-centric data saved successfully.")

Starting data processing for site-centric cache...
Training:  CA.SCB  year: 2014
Training:  CA.SCB  year: 2015
Test:  CA.SCB  year: 2016
Training:  DE.Hte  year: 2011
Training:  DE.Hte  year: 2012
Training:  DE.Hte  year: 2013
Training:  DE.Hte  year: 2014
Training:  DE.Hte  year: 2015
Training:  DE.Hte  year: 2016
Training:  DE.Hte  year: 2017
Test:  DE.Hte  year: 2018
Training:  DE.SfN  year: 2012
Training:  DE.SfN  year: 2013
Test:  DE.SfN  year: 2014
Training:  DE.Zrk  year: 2013
Training:  DE.Zrk  year: 2014
Training:  DE.Zrk  year: 2015
Training:  DE.Zrk  year: 2016
Training:  DE.Zrk  year: 2017
Test:  DE.Zrk  year: 2018
Training:  FI.Lom  year: 2006
Training:  FI.Lom  year: 2007
Training:  FI.Lom  year: 2008
Training:  FI.Lom  year: 2009
Test:  FI.Lom  year: 2010
Training:  FI.Si2  year: 2012
Training:  FI.Si2  year: 2013
Training:  FI.Si2  year: 2014
Training:  FI.Si2  year: 2015
Test:  FI.Si2  year: 2016
Training:  FI.Sii  year: 2013
Training:  FI.Sii  year: 2014
Training:  FI

In [31]:
# temporal
train_data_x, train_data_y, test_data_x, test_data_y = [], [], [], []
test_site_list = []
shape_counter = 0
wet_counter = 0
for (site, year), group in grouped:
    if site not in location.keys():
        continue
    lat, lon, elev, IGBP, mat, map = location[site]
    if group.shape[0] != 365:
        print("shape: ", site)
        shape_counter += 1
        continue
    if IGBP != 'WET':
        print("wet:", site)
        wet_counter += 1
        continue
    temp = group.to_numpy()[:,-6:-3]
    temp = temp.astype(dtype=float)
    lat_ind, lon_ind = nearest_patch_point(lat, lon)
    print(site, year, lat_ind, lon_ind)
    for i, feat in enumerate(stat_features):
        if i == 0:
            input = np.tile(elev, (365))[..., np.newaxis]
        elif feat == 'wetlandtype':
            if site in site_wet.keys():
                wet = site_wet[site]
            else:
                wet = stat_dict[feat][lat_ind, lon_ind]
            wet = np.tile(wet, (365))[..., np.newaxis]
            input = np.concatenate((input, wet), axis=1)
        else:
            new_input = stat_dict[feat][lat_ind, lon_ind]
            new_input = np.tile(new_input, (365))[..., np.newaxis]
            input = np.concatenate((input, new_input), axis=1)
    ch4_value = np.tile(ch4_dict[year], (365))[..., np.newaxis]
    co2_value = np.tile(co2_dict[year], (365))[..., np.newaxis]
    input = np.concatenate((input, ch4_value), axis=1)
    input = np.concatenate((input, co2_value), axis=1)

    input = np.concatenate((input, temp[:,2][..., np.newaxis]), axis=1) #prec

    # solr 
    input_file = f'../../data/TEM-MDM/daily_ecmwf_SOLR_{year}.nc'
    feature = convert2year(xr.open_dataset(input_file, engine="netcdf4")['SOLR'].values[lat_ind, lon_ind,:,:].reshape(1,1,12,31))
    input = np.concatenate((input, feature.reshape(365,1)), axis=1)

    input = np.concatenate((input, temp[:,0][..., np.newaxis]), axis=1) # tair
    
    T = group['TA_F'].to_numpy()
    VPD = group['VPD_F'].to_numpy()
    vp = (0.6108 * np.exp(17.27*T/(T+237.3)))*10 - VPD
    input = np.concatenate((input, vp[..., np.newaxis]), axis=1) # vapr

    # npp 
    input_file = f'../../data/TEM-MDM/monthly_NPP_{year}.nc'
    feature = xr.open_dataset(input_file, engine="netcdf4")['NPP'].values[lat_ind, lon_ind,:].reshape(1,1,12)
    feature = np.nan_to_num(feature, nan=0.0)
    feature = np.repeat(feature[:, :, :, np.newaxis], 31, axis=3)
    feature = convert2year(feature)
    input = np.concatenate((input, feature.reshape(365,1)), axis=1)
    
    if np.isnan(input).any():
        print('Input nan')
        continue
    label = group['FCH4_F'].to_numpy()
    mask = np.isnan(label)
    label[mask] = group['FCH4_F_ANNOPTLM'].to_numpy()[mask]
    if year > splitting_year_dict[site]:
        print("test site: ", site, " year:", year)
        test_data_x.append(input)
        test_data_y.append(label)
        test_site_list.append(site)
    else:
        print("training site: ", site, " year:", year)
        train_data_x.append(input)
        train_data_y.append(label)

print(shape_counter)
print(wet_counter)

wet: AT.Neu
wet: AT.Neu
wet: AT.Neu
shape:  BR.Npw
wet: BR.Npw
wet: BR.Npw
wet: BR.Npw
BW.Gum 2018 142 405
training site:  BW.Gum  year: 2018
wet: BW.Nxr
CA.SCB 2014 303 117
training site:  CA.SCB  year: 2014
CA.SCB 2015 303 117
training site:  CA.SCB  year: 2015
CA.SCB 2016 303 117
training site:  CA.SCB  year: 2016
shape:  CA.SCB
wet: CA.SCC
wet: CA.SCC
wet: CA.SCC
shape:  CA.SCC
wet: CH.Cha
wet: CH.Cha
wet: CH.Cha
wet: CH.Cha
wet: CH.Cha
wet: CH.Dav
wet: CH.Dav
wet: CH.Oe2
shape:  CN.Hgu
wet: CN.Hgu
wet: CN.Hgu
wet: DE.Dgw
wet: DE.Dgw
wet: DE.Dgw
wet: DE.Dgw
DE.Hte 2011 288 384
training site:  DE.Hte  year: 2011
DE.Hte 2012 288 384
training site:  DE.Hte  year: 2012
DE.Hte 2013 288 384
training site:  DE.Hte  year: 2013
DE.Hte 2014 288 384
training site:  DE.Hte  year: 2014
DE.Hte 2015 288 384
training site:  DE.Hte  year: 2015
DE.Hte 2016 288 384
training site:  DE.Hte  year: 2016
DE.Hte 2017 288 384
training site:  DE.Hte  year: 2017
DE.Hte 2018 288 384
test site:  DE.Hte  year: 2

In [32]:
train_data_x = np.stack(train_data_x, axis=0)
train_data_y = np.stack(train_data_y, axis=0)
test_data_x = np.stack(test_data_x, axis=0)
test_data_y = np.stack(test_data_y, axis=0)
print(train_data_x[0][0][0])
print(train_data_x[3][0][0])
print(train_data_x[6][0][0])
print(train_data_x[7][0][0])
print(train_data_x[8][0][0])
print(train_data_x[11][0][0])

950.0
280.0
0.2
0.2
0.2
590.0


In [33]:
train_data_x.shape, train_data_y.shape, test_data_x.shape, test_data_y.shape

((71, 365, 15), (71, 365), (17, 365, 15), (17, 365))

In [17]:
np.save('../../processed_data/FLUXNET-CH4/temporal/train_data_x.npy', train_data_x)
np.save('../../processed_data/FLUXNET-CH4/temporal/train_data_y.npy', train_data_y)
np.save('../../processed_data/FLUXNET-CH4/temporal/test_data_x.npy', test_data_x)
np.save('../../processed_data/FLUXNET-CH4/temporal/test_data_y.npy', test_data_y)

In [18]:
num_folds = 5
folds = {0: ['BW.Gum', 'FI.Sii', 'RU.Ch2', 'US.Ivo', 'US.Srr', 'US.Bes'],
    1: ['CA.SCB', 'FI.Lom', 'US.WPT', 'US.BZF', 'US.LA2', 'US.BZB'],
    2: ['DE.Zrk', 'RU.Che', 'US.MRM', 'US.LA1', 'US.Los', 'US.NC4'],
    3: ['JP.BBY', 'FR.LGt', 'US.ICs', 'US.Atq', 'US.Myb', 'US.OWC'],
    4: ['DE.SfN', 'DE.Hte', 'US.ORv', 'US.DPW', 'FI.Si2', 'US.Beo']}

In [19]:
for fold in range(num_folds):
    test_sites = set(folds[fold]) 
    data_x, data_y = [], []
    for (site, year), group in grouped:
        if site not in test_sites:
            continue
        lat, lon, elev, IGBP, mat, map = location[site]
        if group.shape[0] != 365:
            continue
        if IGBP != 'WET':
            continue
        temp = group.to_numpy()[:,-6:-3]
        temp = temp.astype(dtype=float)
        lat_ind, lon_ind = nearest_patch_point(lat, lon)
        for i, feat in enumerate(stat_features):
            if i == 0:
                input = np.tile(elev, (365))[..., np.newaxis]
            elif feat == 'wetlandtype':
                if site in site_wet.keys():
                    wet = site_wet[site]
                else:
                    wet = stat_dict[feat][lat_ind, lon_ind]
                wet = np.tile(wet, (365))[..., np.newaxis]
                input = np.concatenate((input, wet), axis=1)
            else:
                new_input = stat_dict[feat][lat_ind, lon_ind]
                new_input = np.tile(new_input, (365))[..., np.newaxis]
                input = np.concatenate((input, new_input), axis=1)
        ch4_value = np.tile(ch4_dict[year], (365))[..., np.newaxis]
        co2_value = np.tile(co2_dict[year], (365))[..., np.newaxis]
        input = np.concatenate((input, ch4_value), axis=1)
        input = np.concatenate((input, co2_value), axis=1)

        input = np.concatenate((input, temp[:,2][..., np.newaxis]), axis=1) #prec

        # solr 
        input_file = f'../../data/TEM-MDM/daily_ecmwf_SOLR_{year}.nc'
        feature = convert2year(xr.open_dataset(input_file, engine="netcdf4")['SOLR'].values[lat_ind, lon_ind,:,:].reshape(1,1,12,31))
        input = np.concatenate((input, feature.reshape(365,1)), axis=1)

        input = np.concatenate((input, temp[:,0][..., np.newaxis]), axis=1) # tair
        
        T = group['TA_F'].to_numpy()
        VPD = group['VPD_F'].to_numpy()
        vp = (0.6108 * np.exp(17.27*T/(T+237.3)))*10 - VPD
        input = np.concatenate((input, vp[..., np.newaxis]), axis=1) # vapr

        # npp 
        input_file = f'../../data/TEM-MDM/monthly_NPP_{year}.nc'
        feature = xr.open_dataset(input_file, engine="netcdf4")['NPP'].values[lat_ind, lon_ind,:].reshape(1,1,12)
        feature = np.nan_to_num(feature, nan=0.0)
        feature = np.repeat(feature[:, :, :, np.newaxis], 31, axis=3)
        feature = convert2year(feature)
        input = np.concatenate((input, feature.reshape(365,1)), axis=1)
        
        if np.isnan(input).any():
            continue
        label = group['FCH4_F'].to_numpy()
        mask = np.isnan(label)
        label[mask] = group['FCH4_F_ANNOPTLM'].to_numpy()[mask]
        data_x.append(input)
        data_y.append(label)
        print(site, year, label.shape, np.max(label))
    data_x = np.stack(data_x, axis=0)
    data_y = np.stack(data_y, axis=0)
    print("fold", fold, data_x.shape, data_y.shape, np.max(data_y))
    np.save(f'../../processed_data/FLUXNET-CH4/spatial/data_x_{fold}.npy', data_x)
    np.save(f'../../processed_data/FLUXNET-CH4/spatial/data_y_{fold}.npy', data_y)


BW.Gum 2018 (365,) 483.563322315711
FI.Sii 2013 (365,) 153.515458333333
FI.Sii 2014 (365,) 146.255791666667
FI.Sii 2015 (365,) 116.192875
FI.Sii 2016 (365,) 164.675625
FI.Sii 2017 (365,) 132.359145833333
FI.Sii 2018 (365,) 79.9650833333333
RU.Ch2 2014 (365,) 97.114675
RU.Ch2 2015 (365,) 82.3404791666667
RU.Ch2 2016 (365,) 81.35925
US.Ivo 2013 (365,) 42.0245416666667
US.Ivo 2014 (365,) 47.6107708333333
US.Ivo 2015 (365,) 36.2935273744241
US.Ivo 2016 (365,) 64.3158125
US.Srr 2014 (365,) 10.87000125
US.Srr 2015 (365,) 10.2566639583333
US.Srr 2016 (365,) 12.7582220833333
fold 0 (17, 365, 15) (17, 365) 483.563322315711
CA.SCB 2014 (365,) 106.205601875
CA.SCB 2015 (365,) 107.18146
CA.SCB 2016 (365,) 113.58094625
FI.Lom 2006 (365,) 127.904755666667
FI.Lom 2007 (365,) 217.510894104167
FI.Lom 2008 (365,) 170.345764395833
FI.Lom 2009 (365,) 184.903329416667
FI.Lom 2010 (365,) 131.315445520833
US.BZB 2014 (365,) 84.4521958333333
US.BZB 2015 (365,) 113.388708333333
US.BZB 2016 (365,) 229.113002083